**Sel 1. Import Semua Library**

In [3]:
!pip install textblob pandas

In [4]:
# ==========================================
# SEL 1: IMPORT SEMUA LIBRARY DI AWAL
# ==========================================
import pandas as pd
import numpy as np
import re
from textblob import TextBlob

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

# Library untuk mengatasi Imbalance Data
from imblearn.over_sampling import SMOTE

# Library untuk Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

print("Semua library berhasil diimpor!")

Semua library berhasil diimpor!


- Apa yang dilakukan: Sel ini mengumpulkan semua "alat tempur" (library) yang
kita butuhkan dari atas sampai bawah notebook ke dalam satu tempat.

- Kenapa dilakukan? Ini adalah standar penulisan kode (best practice) di industri dan secara spesifik. Dengan menaruhnya di atas, program tidak perlu bolak-balik mencari alat di tengah proses, dan orang yang membaca kode langsung tahu apa saja yang dibutuhkan.



**Sel 2. Membaca & Membersihkan Data**

In [5]:
# ==========================================
# SEL 2: PREPROCESSING & PELABELAN
# ==========================================
df = pd.read_csv('dataset_komentar_youtube.csv')

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

df['cleaned_text'] = df['text'].apply(clean_text)
df = df[df['cleaned_text'].str.strip() != '']

def get_sentiment(text):
    analysis = TextBlob(text)
    if analysis.sentiment.polarity > 0.05:
        return 'positif'
    elif analysis.sentiment.polarity < -0.05:
        return 'negatif'
    else:
        return 'netral'

df['label'] = df['cleaned_text'].apply(get_sentiment)

print("Distribusi kelas sebelum SMOTE:")
print(df['label'].value_counts())

Distribusi kelas sebelum SMOTE:
label
netral     7663
positif    2481
negatif    1063
Name: count, dtype: int64


- Apa yang dilakukan: Pertama, kita membaca file CSV dataset. Lalu, fungsi clean_text membuang semua "sampah" pada teks (seperti link URL, angka, tanda baca, emoji) dan mengubah huruf menjadi kecil semua (lowercase). Setelah bersih, fungsi get_sentiment (memakai TextBlob) otomatis membaca emosi kata-kata bahasa Inggris dan memberinya label: Positif, Netral, atau Negatif.

- Kenapa dilakukan? Model AI itu seperti anak kecil yang belajar membaca; kalau teksnya penuh simbol aneh dan campur aduk, dia akan pusing. Teks harus benar-benar bersih agar model bisa fokus mencari kata kunci sentimen.

**Sel 3. Ekstraksi Fitur & Penanganan Imbalance (SMOTE)**

> Tambahkan blockquote



In [6]:
# ==========================================
# SEL 3: SPLITTING & SMOTE (OVERSAMPLING)
# ==========================================
# Mengubah teks (negatif, netral, positif) jadi angka (0, 1, 2)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(df['label'])
X = df['cleaned_text'].astype(str)

# Membagi data 80/20 untuk semua skema agar konsisten
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# Ekstraksi Fitur dengan TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Menerapkan SMOTE untuk menyeimbangkan kelas pada data training
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_tfidf, y_train)

print(f"Jumlah data training tiap kelas SEBELUM SMOTE: {np.bincount(y_train)}")
print(f"Jumlah data training tiap kelas SESUDAH SMOTE: {np.bincount(y_train_smote)}")

Jumlah data training tiap kelas SEBELUM SMOTE: [ 853 6110 2002]
Jumlah data training tiap kelas SESUDAH SMOTE: [6110 6110 6110]


- Apa yang dilakukan: 1. Mengubah label huruf ("positif", "negatif", "netral") menjadi angka (0, 1, 2) karena AI hanya paham angka.
2. Membagi data menjadi 80% untuk training (materi belajar AI) dan 20% untuk testing (soal ujian AI).
3. Mengubah kata-kata menjadi bobot angka menggunakan algoritma TF-IDF.
4. (Penting): Menjalankan SMOTE pada data training.

- Kenapa dilakukan? Ini adalah solusi langsung dari soal distribusi tidak seimbang (imbalance). Karena data Netral kamu sangat banyak (misal 7000), sedangkan Negatif hanya sedikit (misal 1000), SMOTE akan secara cerdas menciptakan "data kloningan/sintetis" untuk yang Negatif dan Positif hingga jumlahnya sama-sama 7000. Hasilnya, AI belajar dengan adil tanpa berat sebelah.

**Sel 4. Skema Machine Learning (SVM & Random Forest)**

In [7]:
# ==========================================
# SEL 4: PELATIHAN ML KLASIK (SKEMA 2 & 3)
# ==========================================

print("--- Menjalankan Skema 2: SVM ---")
svm_model = SVC(kernel='linear')
svm_model.fit(X_train_smote, y_train_smote) # Melatih pakai data yang sudah di-SMOTE
svm_pred = svm_model.predict(X_test_tfidf)
print(f"Akurasi SVM: {accuracy_score(y_test, svm_pred) * 100:.2f}%\n")

print("--- Menjalankan Skema 3: Random Forest ---")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_smote, y_train_smote)
rf_pred = rf_model.predict(X_test_tfidf)
print(f"Akurasi Random Forest: {accuracy_score(y_test, rf_pred) * 100:.2f}%\n")

--- Menjalankan Skema 2: SVM ---
Akurasi SVM: 94.51%

--- Menjalankan Skema 3: Random Forest ---
Akurasi Random Forest: 91.44%



- Apa yang dilakukan: Sel ini melatih dua algoritma Machine Learning tradisional, yaitu Support Vector Machine (SVM) dan Random Forest. Kita melatihnya menggunakan X_train_smote (data yang sudah diseimbangkan jumlahnya tadi). Setelah belajar, model langsung disuruh mengerjakan soal ujian (X_test_tfidf) dan dihitung akurasinya.

- Kenapa dilakukan? Untuk memenuhi syarat wajib 3 skema eksperimen dari Dicoding, sekaligus membandingkan mana algoritma yang paling cocok untuk data.

**Sel 5. Skema Deep Learning (Memperbaiki Variabel Error)**

In [8]:
# ==========================================
# SEL 5: PELATIHAN DEEP LEARNING (SKEMA 1)
# ==========================================

print("--- Menjalankan Skema 1: Deep Learning (DNN) ---")

# INI SOLUSI ERROR-NYA: Kita definisikan y_train_dl dan y_test_dl menggunakan to_categorical
y_train_dl = to_categorical(y_train_smote, num_classes=3)
y_test_dl = to_categorical(y_test, num_classes=3)

# Ubah TF-IDF menjadi Dense Array untuk Keras
X_train_dl = X_train_smote.toarray()
X_test_dl = X_test_tfidf.toarray()

model_dl = Sequential([
    Dense(256, activation='relu', input_shape=(X_train_dl.shape[1],)),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

model_dl.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Melatih model (Pakai GPU Colab agar cepat)
history = model_dl.fit(
    X_train_dl, y_train_dl,
    epochs=10,
    batch_size=64,
    validation_data=(X_test_dl, y_test_dl),
    verbose=1
)

loss, accuracy = model_dl.evaluate(X_test_dl, y_test_dl, verbose=0)
print(f"\nAkurasi Deep Learning: {accuracy * 100:.2f}%")

--- Menjalankan Skema 1: Deep Learning (DNN) ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
287/287 ━━━━━━━━━━━━━━━━━━━━ 17s 52ms/step - accuracy: 0.8563 - loss: 0.4016 - val_accuracy: 0.9202 - val_loss: 0.2630
Epoch 2/10
287/287 ━━━━━━━━━━━━━━━━━━━━ 10s 33ms/step - accuracy: 0.9792 - loss: 0.0684 - val_accuracy: 0.9260 - val_loss: 0.2880
Epoch 3/10
287/287 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.9899 - loss: 0.0320 - val_accuracy: 0.9255 - val_loss: 0.3224
Epoch 4/10
287/287 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step - accuracy: 0.9934 - loss: 0.0201 - val_accuracy: 0.9282 - val_loss: 0.3447
Epoch 5/10
287/287 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.9949 - loss: 0.0150 - val_accuracy: 0.9246 - val_loss: 0.3762
Epoch 6/10
287/287 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step - accuracy: 0.9954 - loss: 0.0123 - val_accuracy: 0.9215 - val_loss: 0.3979
Epoch 7/10
287/287 ━━━━━━━━━━━━━━━━━━━━ 7s 25ms/step - accuracy: 0.9957 - loss: 0.0114 - val_accuracy: 0.9170 - val_loss: 0.4178
Epoch 8/10
287/287 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step - accuracy: 0.9969 - loss: 0.0092 - val_ac

- Apa yang dilakukan: Ini adalah model AI tercanggih kita (Deep Neural Network). Di sinilah error sebelumnya diperbaiki. Kita mendefinisikan y_train_dl menggunakan to_categorical (mengubah angka 0,1,2 jadi format matriks 3 kolom) yang merupakan syarat mutlak agar Deep Learning bisa berjalan. Model kemudian dibangun menggunakan lapisan Dense (neuron otak) dan Dropout (fitur pelupa agar model tidak cuma menghafal data).

- Kenapa dilakukan? Ini adalah "senjata utama" kita untuk menembus target nilai akurasi di atas 92%.

**Sel 6. Pengujian Inferensi**

In [9]:
# ==========================================
# SEL 6: INFERENSI (TESTING KALIMAT BARU)
# ==========================================
print("=== PENGUJIAN INFERENSI MODEL ===\n")

kalimat_baru = [
    "This game looks absolutely amazing, masterpiece! I can't wait to play it.",
    "The graphics are terrible and the gameplay is so boring, very bad.",
    "When is the release date for pc?"
]

kalimat_bersih = [clean_text(teks) for teks in kalimat_baru]
fitur_baru = tfidf.transform(kalimat_bersih).toarray()

prediksi_probabilitas = model_dl.predict(fitur_baru)
prediksi_kelas = np.argmax(prediksi_probabilitas, axis=1)
hasil_label = label_encoder.inverse_transform(prediksi_kelas)

for i, teks in enumerate(kalimat_baru):
    print(f"Input: '{teks}'")
    print(f"Prediksi Sentimen: {hasil_label[i].upper()}\n")

=== PENGUJIAN INFERENSI MODEL ===

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
Input: 'This game looks absolutely amazing, masterpiece! I can't wait to play it.'
Prediksi Sentimen: POSITIF

Input: 'The graphics are terrible and the gameplay is so boring, very bad.'
Prediksi Sentimen: NEGATIF

Input: 'When is the release date for pc?'
Prediksi Sentimen: NETRAL



=== MODEL INFERENSI PENGUJIAN ===

Input: 'Game ini terlihat sangat menakjubkan, mahakarya! Saya tidak sabar untuk memainkannya.'

Prediksi Sentimen: POSITIF

Input: 'Grafiknya mengerikan dan gameplay-nya sangat membosankan, sangat buruk.'

Prediksi Sentimen: NEGATIF

Input: 'Kapan tanggal rilis untuk PC?'

Prediksi Sentimen: NETRAL

- Apa yang dilakukan: Kita membuat 3 kalimat baru buatan kita sendiri. Kalimat itu lalu dilewatkan ke fungsi pembersih, diubah ke bentuk TF-IDF, lalu kita suruh model Deep Learning kita menebak apa sentimennya (keluarannya berupa prediksi: POSITIF, NEGATIF, NETRAL).

- Kenapa dilakukan? Untuk membuktikan bahwa model kita bukan sekadar angka akurasi yang tinggi di atas kertas, tetapi benar-benar berfungsi di dunia nyata jika dimasukkan kalimat komentar baru.

**Sel 7. Requirements**

In [10]:
# ==========================================
# SEL 7: EXPORT REQUIREMENTS
# ==========================================
requirements_content = """pandas==2.0.3
scikit-learn==1.2.2
tensorflow==2.15.0
textblob==0.17.1
youtube-comment-downloader==0.3.0
imbalanced-learn==0.10.1
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements_content)

print("File requirements.txt berhasil dibuat!")

File requirements.txt berhasil dibuat!


- Apa yang dilakukan: Menuliskan daftar nama library beserta versi spesifiknya (misalnya pandas versi 2.0.3, tensorflow versi 2.15.0) lalu mencetaknya ke dalam file teks bernama requirements.txt.
